# 01. Importando bibliotecas

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder

# 02. Importando el dataset

In [2]:
url = 'https://raw.githubusercontent.com/ezeperezds/Anemia-Prediction-System/refs/heads/develop/data/raw/anemia.csv'

data = pd.read_csv(url)

In [3]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result
760,1,16.6,18.8,28.1,70.9,0
347,1,13.1,25.6,28.4,77.5,1
94,1,11.6,24.3,29.1,85.7,1
1168,0,11.7,24.4,31.5,99.8,1
442,1,14.8,16.3,27.8,76.4,0
487,0,14.7,28.9,31.0,69.8,0
78,1,10.7,19.0,32.2,77.1,1
486,0,13.4,25.2,30.2,95.9,0
1086,0,13.3,19.1,30.9,97.9,0
404,1,16.7,16.5,30.5,94.5,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1421 entries, 0 to 1420
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Gender      1421 non-null   int64  
 1   Hemoglobin  1421 non-null   float64
 2   MCH         1421 non-null   float64
 3   MCHC        1421 non-null   float64
 4   MCV         1421 non-null   float64
 5   Result      1421 non-null   int64  
dtypes: float64(4), int64(2)
memory usage: 66.7 KB


# 3. Feature Engineering

## Hemoglobina baja según genero

Si el paciente es hombre y su hemoglobina es menor a 13, tiene la hemoglobina baja

Si el paciente es mujer y su hemoglobina es menor a 12, tiene la hemoglobina baja

In [5]:
anemia_hb = []

for index, row in data.iterrows():
  if row['Hemoglobin'] < 13 and row['Gender'] == 0:
    anemia_hb.append(1)
  elif row['Hemoglobin'] < 12 and row['Gender'] == 1:
    anemia_hb.append(1)
  else:
    anemia_hb.append(0)

In [6]:
data['Low_Hb'] = anemia_hb

In [7]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb
760,1,16.6,18.8,28.1,70.9,0,0
347,1,13.1,25.6,28.4,77.5,1,0
94,1,11.6,24.3,29.1,85.7,1,1
1168,0,11.7,24.4,31.5,99.8,1,1
442,1,14.8,16.3,27.8,76.4,0,0
487,0,14.7,28.9,31.0,69.8,0,0
78,1,10.7,19.0,32.2,77.1,1,1
486,0,13.4,25.2,30.2,95.9,0,0
1086,0,13.3,19.1,30.9,97.9,0,0
404,1,16.7,16.5,30.5,94.5,0,0


## Clasificación del MCV

Si MCV es menor a 80 -> Microcítica

Si MCV esta entre 80 - 100 -> Normocítica (Normal)

Si MCV es mayor a 100 -> Macrocítica

In [8]:
mcv_categoria = []

for index, row in data.iterrows():
  if (row['MCV'] < 80):
    mcv_categoria.append('Micro')
  elif ((row['MCV'] >= 80) & (row['MCV']<=100)):
    mcv_categoria.append('Normal')
  else:
    mcv_categoria.append('Macro')

In [9]:
data['MCV_Cat'] = mcv_categoria

In [10]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat
760,1,16.6,18.8,28.1,70.9,0,0,Micro
347,1,13.1,25.6,28.4,77.5,1,0,Micro
94,1,11.6,24.3,29.1,85.7,1,1,Normal
1168,0,11.7,24.4,31.5,99.8,1,1,Normal
442,1,14.8,16.3,27.8,76.4,0,0,Micro
487,0,14.7,28.9,31.0,69.8,0,0,Micro
78,1,10.7,19.0,32.2,77.1,1,1,Micro
486,0,13.4,25.2,30.2,95.9,0,0,Normal
1086,0,13.3,19.1,30.9,97.9,0,0,Normal
404,1,16.7,16.5,30.5,94.5,0,0,Normal


## Relación entre Hemoglobina y MCV

In [11]:
data['HB_MCV_ratio'] = (data['Hemoglobin'] / data['MCV']).round(2)

In [12]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18


## Clasificación del MCHC

Si MCHC es menor a 32 -> Bajo

Si MCHC esta entre 32 y 36 -> Normal

Si MCHC es mayor a 36 -> Alto

In [13]:
mchc_cat = []

for index, row in data.iterrows():
  if (row['MCHC']) < 32:
    mchc_cat.append('Bajo')
  elif ((row['MCHC'] >= 32) & (row['MCHC'] <= 36)):
    mchc_cat.append('Normal')
  else:
    mchc_cat.append('Alto')

In [14]:
data['MCHC_Cat'] = mchc_cat

In [15]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio,MCHC_Cat
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23,Bajo
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17,Bajo
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14,Bajo
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12,Bajo
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19,Bajo
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21,Bajo
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14,Normal
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14,Bajo
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14,Bajo
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18,Bajo


## Clasificación del MCH

Si MCH es menor a 27 -> Bajo

Si MCH esta entre 27 y 33 -> Normal

Si MCH es mayor a 33 -> Alto

In [16]:
mch_cat = []

for index, row in data.iterrows():
  if (row['MCH']) < 27:
    mch_cat.append('Bajo')
  elif ((row['MCH'] >= 27) & (row['MCH'] <= 33)):
    mch_cat.append('Normal')
  else:
    mch_cat.append('Alto')

In [17]:
data['MCH_Cat'] = mch_cat

In [18]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio,MCHC_Cat,MCH_Cat
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23,Bajo,Bajo
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17,Bajo,Bajo
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14,Bajo,Bajo
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12,Bajo,Bajo
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19,Bajo,Bajo
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21,Bajo,Normal
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14,Normal,Bajo
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14,Bajo,Bajo
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14,Bajo,Bajo
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18,Bajo,Bajo


## Indicador de microcitosis

In [19]:
data['Microcitosis'] = np.where(data['MCV'] < 80, 1, 0)

In [20]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio,MCHC_Cat,MCH_Cat,Microcitosis
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23,Bajo,Bajo,1
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17,Bajo,Bajo,1
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14,Bajo,Bajo,0
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12,Bajo,Bajo,0
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19,Bajo,Bajo,1
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21,Bajo,Normal,1
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14,Normal,Bajo,1
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14,Bajo,Bajo,0
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14,Bajo,Bajo,0
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18,Bajo,Bajo,0


## Indicador de hipocromía

In [21]:
data['Hipocromía'] = np.where(data['MCHC'] < 32, 1, 0)

In [22]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio,MCHC_Cat,MCH_Cat,Microcitosis,Hipocromía
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23,Bajo,Bajo,1,1
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17,Bajo,Bajo,1,1
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14,Bajo,Bajo,0,1
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12,Bajo,Bajo,0,1
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19,Bajo,Bajo,1,1
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21,Bajo,Normal,1,1
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14,Normal,Bajo,1,0
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14,Bajo,Bajo,0,1
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14,Bajo,Bajo,0,1
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18,Bajo,Bajo,0,1


## Score hematológico simple

In [23]:
data['Score'] = data['Low_Hb'] + (
    (data['MCHC_Cat'] == 'Bajo').astype(int) +
    (data['MCH_Cat'] == 'Bajo').astype(int) +
    (data['MCV_Cat'] == 'Micro').astype(int)
)

In [24]:
data.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Result,Low_Hb,MCV_Cat,HB_MCV_ratio,MCHC_Cat,MCH_Cat,Microcitosis,Hipocromía,Score
760,1,16.6,18.8,28.1,70.9,0,0,Micro,0.23,Bajo,Bajo,1,1,3
347,1,13.1,25.6,28.4,77.5,1,0,Micro,0.17,Bajo,Bajo,1,1,3
94,1,11.6,24.3,29.1,85.7,1,1,Normal,0.14,Bajo,Bajo,0,1,3
1168,0,11.7,24.4,31.5,99.8,1,1,Normal,0.12,Bajo,Bajo,0,1,3
442,1,14.8,16.3,27.8,76.4,0,0,Micro,0.19,Bajo,Bajo,1,1,3
487,0,14.7,28.9,31.0,69.8,0,0,Micro,0.21,Bajo,Normal,1,1,2
78,1,10.7,19.0,32.2,77.1,1,1,Micro,0.14,Normal,Bajo,1,0,3
486,0,13.4,25.2,30.2,95.9,0,0,Normal,0.14,Bajo,Bajo,0,1,2
1086,0,13.3,19.1,30.9,97.9,0,0,Normal,0.14,Bajo,Bajo,0,1,2
404,1,16.7,16.5,30.5,94.5,0,0,Normal,0.18,Bajo,Bajo,0,1,2


# 4. Encoding

In [25]:
# Segmentación de variables explicativas y variable objetivo
X = data.drop(columns='Result')
y = data['Result']

In [26]:
columnas_categoricas = ['MCV_Cat', 'MCHC_Cat', 'MCH_Cat']

columnas_numericas = ['Gender', 'Hemoglobin', 'MCH',
                      'MCHC', 'MCV', 'Low_Hb',
                      'HB_MCV_ratio','Microcitosis', 'Hipocromía', 'Score'
                      ]

In [27]:
encoder = OneHotEncoder(drop='first', sparse_output=False)

In [28]:
columnas_cat = encoder.fit_transform(X[columnas_categoricas])

In [29]:
df_cat = pd.DataFrame(columnas_cat,
                      columns=encoder.get_feature_names_out(columnas_categoricas)
                      )

df_cod = pd.concat([X[columnas_numericas].reset_index(drop=True),
                    df_cat.reset_index(drop=True)],
                   axis=1)

df_cod['Result'] = y

In [30]:
df_cod.sample(10, random_state=82)

,Gender,Hemoglobin,MCH,MCHC,MCV,Low_Hb,HB_MCV_ratio,Microcitosis,Hipocromía,Score,MCV_Cat_Micro,MCV_Cat_Normal,MCHC_Cat_Normal,MCH_Cat_Normal,Result
760,1,16.6,18.8,28.1,70.9,0,0.23,1,1,3,1.0,0.0,0.0,0.0,0
347,1,13.1,25.6,28.4,77.5,0,0.17,1,1,3,1.0,0.0,0.0,0.0,1
94,1,11.6,24.3,29.1,85.7,1,0.14,0,1,3,0.0,1.0,0.0,0.0,1
1168,0,11.7,24.4,31.5,99.8,1,0.12,0,1,3,0.0,1.0,0.0,0.0,1
442,1,14.8,16.3,27.8,76.4,0,0.19,1,1,3,1.0,0.0,0.0,0.0,0
487,0,14.7,28.9,31.0,69.8,0,0.21,1,1,2,1.0,0.0,0.0,1.0,0
78,1,10.7,19.0,32.2,77.1,1,0.14,1,0,3,1.0,0.0,1.0,0.0,1
486,0,13.4,25.2,30.2,95.9,0,0.14,0,1,2,0.0,1.0,0.0,0.0,0
1086,0,13.3,19.1,30.9,97.9,0,0.14,0,1,2,0.0,1.0,0.0,0.0,0
404,1,16.7,16.5,30.5,94.5,0,0.18,0,1,2,0.0,1.0,0.0,0.0,0
